# Phase 3 — Standardisation and quality checks

## Open Materials Data Pipeline for Hydrogel Bioink Design

**Notebook:** `03_standardisation_and_quality_checks.ipynb`  
**Conceptual phase:** 3 — standardisation and quality checks. Record linkage remains conceptual Phase 4 and should use `04_record_linkage.ipynb` when it is constructed.

This notebook continues the completed Phase 2 checkpoint for Zenodo record [19602891](https://zenodo.org/records/19602891), version `v0.0.2`. It does **not** rebuild Phases 1 or 2. Instead, it verifies the controlled checkpoint, reconciles the Phase 2 source-preserving tables against the original rheology ZIP and image-index CSV, applies only evidence-supported standardisation rules, retains flagged observations and prepares a traceable hand-off to Phase 4.

The dataset concerns phenol-modified alginate and hyaluronic-acid inks in an acellular printing study. Source worksheets and image-index records are **not confirmed physical samples, formulations or replicates**. The 4,600 measurement points are nested source observations, not 4,600 independent materials. No physical-sample, formulation, replicate or rheology-to-image relationship is created in Phase 3.

### Mandatory reproducibility contract

The notebook is designed around the following lifecycle:

**save → close Jupyter → reopen Jupyter → open notebook → Restart Kernel → Run All → complete from top to bottom**

A fresh kernel is assumed to contain no variables from an earlier session. Every import, helper function, path, DataFrame, transformation rule and validation object used later is created by an earlier cell in the same execution. Phase 3 outputs are recreated deterministically from the verified Phase 2 checkpoint; the notebook does not append duplicate records or use previous Phase 3 outputs as hidden inputs.

### Input contract and content-addressed file resolution

The preferred repository layout remains:

* `data/raw/zenodo_19602891/`: `ALG-Ph_HA-Ph_rheology_data.zip` and `image_index.csv`.
* `data/metadata/zenodo_source_audit/`: eleven CSV exports and `audit_run_manifest.json`.
* `data/metadata/schema_and_quality_control/`: twenty-six CSV exports and `schema_qc_run_manifest.json`.

However, Phase 3 no longer requires Jupyter to start inside an exact repository folder. The setup cell first locates a valid Phase 2 manifest, then resolves every Phase 1 export, Phase 2 export and raw source file by the SHA-256 recorded in the manifests. Browser/download suffixes such as `(1)`, `(2)` or `(3)` are therefore tolerated **only when the file bytes match the canonical expected hash**. A similarly named but altered file is rejected. This fixes the uploaded run in which the files existed outside the exact repository tree.

Automatic discovery searches the current Jupyter folder and common user folders such as Desktop, Downloads, Documents and OneDrive locations. If the files are stored elsewhere, `INPUT_SEARCH_ROOT_OVERRIDE` in the first code cell can point to their containing folder; this changes only where the notebook looks, never the expected hashes or scientific rules.

Use the documented project kernel, then **Restart Kernel and Run All**. Runtime dependencies are pandas, NumPy, openpyxl and IPython, plus Jupyter/ipykernel to run the notebook. No package-installation command, network call or API key is used in the scientific analysis cells. The notebook records the actual runtime versions used for each execution.

If a canonical repository is detected, Phase 3 outputs are written to `data/processed/standardisation_and_quality_checks/`. Otherwise a portable `phase3_outputs/standardisation_and_quality_checks/` folder is created beside the resolved Phase 2 checkpoint. The output location is printed explicitly. Rerunning replaces only the named Phase 3 outputs and never modifies the verified source/checkpoint files. Source-derived generated tables should remain local review products while dataset redistribution rights remain unresolved.

### Scope and scientific boundaries

Supported identity mappings retain the recorded representations of quantities such as Pa, s, Hz, °C and 1/s. Explicit percentage strain is converted to strain fraction by dividing by 100. Source-labelled fractional strain is retained without clipping or reinterpretation. The viscosity token `mPas` remains unresolved and is **not converted to Pa·s**. Text annotations remain literal, and blanks remain unknown. A valid source date may receive an additional ISO-format representation, but no scientific event meaning is assigned to that date.

No curve fitting, LVR selection, constitutive modelling, representative-modulus extraction, formulation ranking, machine learning, train/test splitting, record linkage, Materials Project modelling or Matbench modelling is performed in Phase 3.

### Reading the notebook

Each scientific section explains:

* the data-science concept;
* the relevant polymer/hydrogel/rheology concept;
* why the step is necessary;
* what the Python code does;
* how to interpret the output;
* what the result can establish;
* what it cannot establish.

A passed structural check means that a stated software/data contract holds. It does not certify instrument accuracy, physical sample identity, annotation meaning, biological performance or model readiness. QC warnings preserve observations for review rather than deleting, imputing, clipping or repairing them.

### Status of the uploaded run and this revision

The uploaded HTML shows that the notebook again stopped in the first setup section: the locator could not find the exact canonical repository tree, so execution ended before the Phase 3 processing cells. The failed HTML should **not** be treated as evidence that the later transformations ran successfully in that Jupyter session.

This version removes that unnecessary coupling between scientific identity and local folder layout. It can resolve canonical filenames or browser-suffixed copies by the hashes already recorded in the Phase 1/2 manifests. It still deliberately stops when a required hash-pinned file is absent, altered, or when scientifically different Phase 2 checkpoints are present. The completed Phase 2 checkpoint provides expected invariants such as 92 worksheets, 4,600 measurement points, 39,100 source measurement cells, 1,031 image-index records, 43 Phase 2 validation rows and 24 retained Phase 2 QC finding records. These are validation expectations, not numbers that the code is allowed to manufacture.

The final acceptance test remains your actual local Jupyter lifecycle: **Restart Kernel → Run All**, save and close Jupyter, reopen it, then **Restart Kernel → Run All** again. A successful scientific workflow should reproduce the same logical CSV outputs from the same verified inputs; the run timestamp in the JSON manifest is expected to change.


## 1. Locate the checkpoint and record the runtime

**Data-science concept.** A portable workflow identifies one controlled checkpoint by content rather than assuming a particular local folder layout. A reproducible notebook must not depend on the Jupyter working directory or browser-added filename suffixes.

**Materials and rheology concept.** Measured properties depend on the acquisition context attached to the source records. Reading the wrong copy of a workbook, checkpoint or manifest can attach the wrong context to a rheological response even when filenames look similar.

**Why this matters.** The uploaded run still failed because the revised locator required an exact repository tree. Your Jupyter session did not expose the files in that exact structure. File location and scientific identity are separate problems: the new resolver uses filenames only to find candidates and SHA-256 to decide whether a candidate is the controlled input.

**What the code does.** It checks required Python packages without installing anything; locates a scientifically compatible Phase 2 manifest; accepts canonical or browser-suffixed candidate filenames only after content validation; prepares controlled search roots for the remaining hash-pinned dependencies; initialises the validation ledger; and records software versions.

**How to interpret the output.** The notebook prints which Phase 2 manifest was selected, how it was found and whether a canonical repository structure was detected. The next section resolves and verifies every remaining dependency by hash, schema and checkpoint count.

**What this can and cannot establish.** Discovery finds candidate files; hash verification establishes byte identity where an upstream expected hash exists. Neither operation proves that the underlying experiment or annotation is scientifically correct.

### Vocabulary: schema, provenance, manifest and hash

A **schema** is a table's contract: what one row represents, which columns exist and what each field means. A **canonical field** is a consistent internal name, such as `storage_modulus_pa`; it is an engineering representation, not a claim that an additional source measurement exists. A **data type** determines whether a token is handled as text, a number, a date or missing information. For example, reading an image identifier as a number could remove a leading zero, so source checkpoint tables are initially read as text.

**Metadata** describes measurements: their units, acquisition axes and source locations. **Provenance** is the history of where a value came from and what happened to it. **Transformation lineage** makes the sequence from original value to standardised representation explicit. Here the original coordinates, scalar and unit are retained even when a standardised value is added.

A **SHA-256 hash** acts like a digital fingerprint of file bytes. If a file changes, its hash normally changes. Even an innocent CSV resave may change bytes and therefore change the hash. Hash equality supports integrity checking; it does not prove that the experimental measurement itself is correct.

A **run manifest** is a machine-readable record of inputs, software versions, decisions, row counts, output hashes and unresolved issues for one execution. Phase 3 verifies hashes that have an upstream expected value. The Phase 2 manifest did not provide an independent hash of itself, so this notebook records its observed hash rather than inventing an unverifiable self-hash; the manifest's exported-file hashes, source/version identity, schemas and checkpoint invariants are then independently checked.

### Polymer context and evidence levels

The supplied study describes phenol-modified alginate (ALG-Ph) and phenol-modified hyaluronic acid (HA-Ph), including mixed inks in aqueous medium. In polymer and hydrogel systems, chain interactions, concentration, molecular architecture, solvent conditions, crosslinking and processing history can affect deformation and flow response. A water-rich polymer network, an uncrosslinked precursor ink and a printed crosslinked construct need not have the same properties. The workflow therefore avoids labelling every rheology row as the property of a finished hydrogel.

The study-level chemistry and nominal methods provide context; they do not authorise assigning a composition, crosslink density, physical sample, replicate or reagent condition to an individual source file without a validated relationship. This separation between **what the source states** and **what is merely plausible** is central to trustworthy materials informatics.

**Permitted assumption:** once the resolved bytes match the recorded hashes and schemas, they represent the intended controlled checkpoint snapshot for this run.  
**Not permitted:** matching bytes prove instrument accuracy, sample independence or complete scientific metadata.


In [ ]:
from pathlib import Path
import csv, io, json, hashlib, platform, zipfile, re, os
from datetime import datetime, timezone
from importlib.metadata import version
from importlib.util import find_spec

# Optional one-time machine settings. Normally leave both as None.
# If automatic discovery cannot see the folder containing your downloaded/project files,
# set INPUT_SEARCH_ROOT_OVERRIDE to that folder. This is a location hint only; every
# controlled input is still accepted by hash, not by filename alone.
INPUT_SEARCH_ROOT_OVERRIDE = None
OUTPUT_ROOT_OVERRIDE = None

required_packages = ['numpy', 'pandas', 'openpyxl', 'IPython']
missing_packages = [name for name in required_packages if find_spec(name) is None]
if missing_packages:
    raise ImportError(
        'Required packages are missing: ' + ', '.join(missing_packages) +
        '. Select the documented project kernel before running this notebook. '
        'No packages were installed by this notebook.'
    )

import numpy as np
import pandas as pd
import openpyxl
from IPython.display import display

EXPECTED_SOURCE_RECORD = 'https://zenodo.org/records/19602891'
EXPECTED_SOURCE_VERSION = 'v0.0.2'
EXPECTED_PHASE2_NAME = 'schema_qc_run_manifest.json'
EXPECTED_PHASE1_NAME = 'audit_run_manifest.json'

checks = []

def check(name, condition, evidence=''):
    ok = bool(condition)
    checks.append({'check': name, 'status': 'PASS' if ok else 'FAIL', 'evidence': str(evidence)})
    if not ok:
        raise ValueError(
            f'{name}: {evidence}. Investigate before continuing; do not bypass this check.'
        )

def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_csv(path):
    return pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        encoding='utf-8-sig'
    )

def unique_paths(paths):
    seen, result = set(), []
    for value in paths:
        if value is None:
            continue
        try:
            p = Path(value).expanduser().resolve()
        except (OSError, RuntimeError):
            continue
        if not p.exists():
            continue
        key = os.path.normcase(str(p))
        if key not in seen:
            seen.add(key)
            result.append(p)
    return result

def canonicalise_download_name(filename):
    """
    Convert a browser-style duplicate name such as image_index(3).csv or
    image_index (3).csv back to image_index.csv for matching purposes only.
    The file is not accepted until its SHA-256 also matches the expected hash.
    """
    p = Path(filename)
    stem = re.sub(r'\s*\(\d+\)$', '', p.stem)
    return stem + p.suffix

def filename_matches(actual_name, canonical_name):
    return canonicalise_download_name(actual_name).casefold() == canonical_name.casefold()

SKIP_DIRS = {
    '.git', '.cache', '.conda', '.venv', 'venv', 'env', 'node_modules',
    'AppData', '$RECYCLE.BIN', 'System Volume Information', '__pycache__'
}

def walk_matching(base, canonical_name, max_depth=6):
    """Yield exact-name or browser-suffixed candidates without reading unrelated file bytes."""
    base = Path(base)
    if not base.is_dir():
        return
    try:
        base_resolved = base.resolve()
    except (OSError, RuntimeError):
        return
    for root, dirs, files in os.walk(base_resolved, topdown=True, onerror=lambda _e: None):
        root_path = Path(root)
        try:
            depth = len(root_path.relative_to(base_resolved).parts)
        except ValueError:
            continue
        dirs[:] = [
            d for d in dirs
            if d not in SKIP_DIRS and not d.startswith('.')
        ]
        if depth >= max_depth:
            dirs[:] = []
        for filename in files:
            if filename_matches(filename, canonical_name):
                yield root_path / filename

def default_search_roots():
    explicit = INPUT_SEARCH_ROOT_OVERRIDE
    env_root = os.environ.get('HYDROGEL_BIOINK_INPUT_DIR', '').strip()
    if explicit is not None:
        roots = unique_paths([explicit])
        if not roots:
            raise FileNotFoundError(
                f'INPUT_SEARCH_ROOT_OVERRIDE does not exist or is inaccessible: {explicit}'
            )
        return roots, 'INPUT_SEARCH_ROOT_OVERRIDE'
    if env_root:
        roots = unique_paths([env_root])
        if not roots:
            raise FileNotFoundError(
                f'HYDROGEL_BIOINK_INPUT_DIR does not exist or is inaccessible: {env_root}'
            )
        return roots, 'HYDROGEL_BIOINK_INPUT_DIR'

    cwd = Path.cwd().resolve()
    home = Path.home()
    roots = unique_paths([
        cwd,
        home / 'Desktop',
        home / 'Downloads',
        home / 'Documents',
        home / 'GitHub',
        home / 'Documents' / 'GitHub',
        home / 'source' / 'repos',
        home / 'OneDrive',
        home / 'OneDrive' / 'Desktop',
        home / 'OneDrive' / 'Documents',
    ])
    return roots, 'automatic search of current/common user folders'

def manifest2_signature(m):
    # Ignore only run-time fields that may legitimately change across deterministic reruns.
    # All scientific/checkpoint identity fields remain part of the comparison.
    stable = {k: v for k, v in m.items() if k not in {'run_utc', 'environment'}}
    return json.dumps(stable, sort_keys=True, ensure_ascii=False)

def locate_phase2_manifest(search_roots):
    candidates = []
    for root in search_roots:
        candidates.extend(walk_matching(root, EXPECTED_PHASE2_NAME, max_depth=6) or [])
    candidates = unique_paths(candidates)

    valid = []
    rejected = []
    required_keys = {
        'phase', 'source_record', 'source_version', 'phase1_manifest_sha256',
        'verified_input_sha256', 'rows', 'export_sha256', 'environment'
    }
    for path in candidates:
        try:
            data = json.loads(path.read_text(encoding='utf-8-sig'))
        except Exception as exc:
            rejected.append((path, f'not readable JSON: {exc}'))
            continue
        problems = []
        if not required_keys.issubset(data):
            problems.append('missing required keys')
        if data.get('phase') != '2: schema and QC prototype':
            problems.append('wrong phase')
        if data.get('source_record') != EXPECTED_SOURCE_RECORD:
            problems.append('wrong source record')
        if data.get('source_version') != EXPECTED_SOURCE_VERSION:
            problems.append('wrong source version')
        if problems:
            rejected.append((path, '; '.join(problems)))
        else:
            valid.append((path, data, sha(path)))

    if not valid:
        searched = '\n- '.join(str(p) for p in search_roots)
        found = '\n'.join(f'- {p}: {reason}' for p, reason in rejected[:10]) or '- none'
        raise FileNotFoundError(
            'Could not locate a valid Phase 2 schema_qc_run_manifest.json checkpoint.\n'
            'The notebook accepts exact filenames or browser suffixes such as (1), but only '
            'after validating the manifest content.\nSearch roots:\n- ' + searched +
            '\nRejected candidates:\n' + found +
            '\nIf your files are elsewhere, set INPUT_SEARCH_ROOT_OVERRIDE in this cell to the folder '
            'that contains the project/downloaded Phase 1 and Phase 2 files.'
        )

    signatures = {manifest2_signature(data) for _, data, _ in valid}
    if len(signatures) > 1:
        details = '\n- '.join(str(path) for path, _, _ in valid)
        raise RuntimeError(
            'More than one scientifically different Phase 2 checkpoint was found. The notebook '
            'will not choose between them. Keep/use one checkpoint or set INPUT_SEARCH_ROOT_OVERRIDE '
            'to the intended folder. Candidates:\n- ' + details
        )

    cwd = Path.cwd().resolve()
    def rank(item):
        path, _data, _hash = item
        exact_penalty = 0 if path.name == EXPECTED_PHASE2_NAME else 1
        try:
            relative_penalty = 0 if path.is_relative_to(cwd) else 1
        except AttributeError:
            try:
                path.relative_to(cwd); relative_penalty = 0
            except ValueError:
                relative_penalty = 1
        return (exact_penalty, relative_penalty, len(path.parts), str(path).casefold())

    chosen = sorted(valid, key=rank)[0]
    path, data, observed_hash = chosen
    duplicate_count = len(valid) - 1
    method = (
        'valid Phase 2 manifest located by content; '
        + ('canonical filename' if path.name == EXPECTED_PHASE2_NAME else 'browser-suffixed filename')
        + (f'; {duplicate_count} byte/scientifically equivalent additional copy/copies ignored'
           if duplicate_count else '')
    )
    return path, data, observed_hash, method

def dependency_search_roots(manifest_path, base_roots):
    nearby = [manifest_path.parent]
    for parent in list(manifest_path.parents)[:5]:
        # Do not recursively scan a filesystem root such as C:\ or /.
        if parent == Path(parent.anchor):
            continue
        nearby.append(parent)
    return unique_paths([*nearby, *base_roots])

def find_by_expected_hash(canonical_name, expected_hash, roots, max_depth=6):
    candidates = []
    for root in roots:
        candidates.extend(walk_matching(root, canonical_name, max_depth=max_depth) or [])
    candidates = unique_paths(candidates)

    exact_hash = []
    mismatches = []
    for path in candidates:
        actual = sha(path)
        if actual == expected_hash:
            exact_hash.append((path, actual))
        else:
            mismatches.append((path, actual))

    if not exact_hash:
        if mismatches:
            observed = '\n'.join(f'- {p.name}: {h}' for p, h in mismatches[:10])
            raise ValueError(
                f'Found candidate file(s) for {canonical_name}, but none matched the expected SHA-256. '
                f'Expected: {expected_hash}\nObserved candidates:\n{observed}\n'
                'Do not edit the expected hash. Restore the checkpoint file that produced the recorded manifest.'
            )
        raise FileNotFoundError(
            f'Required checkpoint input not found: {canonical_name}. The notebook searched exact and '
            'browser-suffixed names in the controlled search roots. If the file is stored elsewhere, '
            'set INPUT_SEARCH_ROOT_OVERRIDE to its containing project/download folder.'
        )

    def rank(item):
        path, _ = item
        return (
            0 if path.name == canonical_name else 1,
            0 if path.parent == Path.cwd().resolve() else 1,
            len(path.parts),
            str(path).casefold()
        )

    chosen_path, actual = sorted(exact_hash, key=rank)[0]
    note = (
        'exact canonical filename; SHA-256 matched'
        if chosen_path.name == canonical_name
        else 'browser-suffixed filename accepted because SHA-256 matched the canonical checkpoint'
    )
    if len(exact_hash) > 1:
        note += f'; {len(exact_hash)-1} additional byte-identical copy/copies ignored'
    return chosen_path, actual, note


def minimal_walk_roots(roots):
    """Remove nested duplicate search roots so each directory tree is scanned at most once."""
    ordered = sorted(unique_paths(roots), key=lambda p: (len(p.parts), str(p).casefold()))
    kept = []
    for path in ordered:
        nested = False
        for parent in kept:
            try:
                path.relative_to(parent)
                nested = True
                break
            except ValueError:
                pass
        if not nested:
            kept.append(path)
    return kept

def build_candidate_index(canonical_names, roots, max_depth=6):
    """
    Scan the controlled search trees once and index only filenames relevant to this run.
    Browser suffixes are normalised only for candidate matching; hashes decide acceptance.
    """
    canonical_lookup = {name.casefold(): name for name in canonical_names}
    index = {name: [] for name in canonical_names}
    for base in minimal_walk_roots(roots):
        if not base.is_dir():
            continue
        try:
            base_resolved = base.resolve()
        except (OSError, RuntimeError):
            continue
        for root, dirs, files in os.walk(base_resolved, topdown=True, onerror=lambda _e: None):
            root_path = Path(root)
            try:
                depth = len(root_path.relative_to(base_resolved).parts)
            except ValueError:
                continue
            dirs[:] = [d for d in dirs if d not in SKIP_DIRS and not d.startswith('.')]
            if depth >= max_depth:
                dirs[:] = []
            for filename in files:
                canonicalised = canonicalise_download_name(filename).casefold()
                canonical = canonical_lookup.get(canonicalised)
                if canonical is not None:
                    index[canonical].append(root_path / filename)
    return {name: unique_paths(paths) for name, paths in index.items()}

def choose_by_expected_hash(canonical_name, expected_hash, candidates):
    """Choose one byte-identical candidate from an already-built filename index."""
    exact_hash, mismatches = [], []
    for path in unique_paths(candidates):
        actual = sha(path)
        if actual == expected_hash:
            exact_hash.append((path, actual))
        else:
            mismatches.append((path, actual))
    if not exact_hash:
        if mismatches:
            observed = '\n'.join(f'- {p.name}: {h}' for p, h in mismatches[:10])
            raise ValueError(
                f'Found candidate file(s) for {canonical_name}, but none matched the expected SHA-256. '
                f'Expected: {expected_hash}\nObserved candidates:\n{observed}\n'
                'Do not edit the expected hash. Restore the checkpoint file that produced the recorded manifest.'
            )
        raise FileNotFoundError(
            f'Required checkpoint input not found: {canonical_name}. Exact and browser-suffixed names '
            'were searched in the controlled input roots. If the file is elsewhere, set '
            'INPUT_SEARCH_ROOT_OVERRIDE to its containing folder.'
        )
    def rank(item):
        path, _ = item
        return (0 if path.name == canonical_name else 1, len(path.parts), str(path).casefold())
    chosen_path, actual = sorted(exact_hash, key=rank)[0]
    note = (
        'exact canonical filename; SHA-256 matched'
        if chosen_path.name == canonical_name
        else 'browser-suffixed filename accepted because SHA-256 matched the canonical checkpoint'
    )
    if len(exact_hash) > 1:
        note += f'; {len(exact_hash)-1} additional byte-identical copy/copies ignored'
    return chosen_path, actual, note

def canonical_logical_path(stage, canonical_name):
    if stage == 'Phase 1 manifest' or stage == 'Phase 1':
        return f'data/metadata/zenodo_source_audit/{canonical_name}'
    if stage == 'Phase 2 manifest' or stage == 'Phase 2':
        return f'data/metadata/schema_and_quality_control/{canonical_name}'
    if stage == 'Raw':
        return f'data/raw/zenodo_19602891/{canonical_name}'
    return canonical_name

def infer_repository_root(manifest_path):
    manifest_path = Path(manifest_path).resolve()
    for candidate in manifest_path.parents:
        expected = candidate / 'data/metadata/schema_and_quality_control'
        try:
            same_folder = expected.resolve() == manifest_path.parent.resolve()
        except OSError:
            same_folder = False
        if same_folder and (candidate / 'notebooks').is_dir():
            return candidate
    return None

SEARCH_ROOTS, SEARCH_MODE = default_search_roots()
manifest2_path, m2, observed_manifest_hash, MANIFEST_DISCOVERY_METHOD = locate_phase2_manifest(SEARCH_ROOTS)
DEPENDENCY_ROOTS = dependency_search_roots(manifest2_path, SEARCH_ROOTS)
REPOSITORY_ROOT = infer_repository_root(manifest2_path)

runtime = pd.DataFrame(
    [{'component': 'Python', 'version': platform.python_version()}] +
    [{'component': name, 'version': version(name)}
     for name in ['pandas', 'numpy', 'openpyxl', 'ipython']]
)

display(runtime)
print('Phase 2 manifest resolved as:', manifest2_path.name)
print('Manifest discovery:', MANIFEST_DISCOVERY_METHOD)
print('Input search mode:', SEARCH_MODE)
print('Canonical repository structure detected:', bool(REPOSITORY_ROOT))
print('No scientific input has been modified.')

## 2. Verify both checkpoint manifests and all dependencies

**Data-science concept.** Content-addressed provenance connects each table to the checkpoint that produced it. File hashes, schemas and row counts test different aspects of integrity: a filename alone is weak evidence; a hash checks exact bytes; a schema checks table structure; row counts check coverage but not values.

**Materials and rheology concept.** Counts describe source observations and metadata records, not independent hydrogel materials. A 39,100-cell source table is a long representation of recorded responses, acquisition conditions and labels; it is not a population of 39,100 independent samples.

**Why this matters.** Phase 3 must not standardise values until it knows that the source-preserving Phase 2 checkpoint is the one intended for continuation.

**What the code does.** It records the observed Phase 2 manifest hash; verifies its phase, source record and source version; resolves the Phase 1 manifest by the hash pinned by Phase 2; resolves every raw file and Phase 1/2 export by its recorded SHA-256 even when the local filename has a browser suffix; reads checkpoint CSVs as text; checks core Phase 2 invariants; and verifies exact expected schemas.

**How to interpret the output.** Every check with an upstream expected hash must pass. The Phase 2 manifest itself is marked `RECORDED_NOT_PREPINNED` because there is no independent upstream self-hash available in the checkpoint; this is an explicit provenance limitation, not a silent assumption.

**What this can and cannot establish.** These checks detect missing files, byte drift, schema drift and unexpected checkpoint counts. They do not prove physical sample identity, correct instrument calibration or scientifically valid annotations.


In [ ]:
required_manifest2_keys = {
    'phase', 'source_record', 'source_version', 'phase1_manifest_sha256',
    'verified_input_sha256', 'rows', 'export_sha256', 'environment'
}
check(
    'Phase 2 manifest required keys',
    required_manifest2_keys.issubset(m2),
    sorted(required_manifest2_keys - set(m2))
)
check('Phase 2 manifest phase identity', m2['phase'] == '2: schema and QC prototype', m2['phase'])
check('Phase 2 source record identity', m2['source_record'] == EXPECTED_SOURCE_RECORD, m2['source_record'])
check('Phase 2 source version identity', m2['source_version'] == EXPECTED_SOURCE_VERSION, m2['source_version'])
check(
    'Phase 2 row/hash key agreement',
    set(m2['rows']) == set(m2['export_sha256']),
    f"rows={len(m2['rows'])}; hashes={len(m2['export_sha256'])}"
)

# Resolve the Phase 1 manifest by the hash pinned inside Phase 2. Browser suffixes are
# acceptable only when the bytes match this upstream expected hash.
manifest1_path, observed_phase1_hash, phase1_manifest_resolution = find_by_expected_hash(
    EXPECTED_PHASE1_NAME,
    m2['phase1_manifest_sha256'],
    DEPENDENCY_ROOTS,
    max_depth=6
)
m1 = json.loads(manifest1_path.read_text(encoding='utf-8-sig'))
check(
    'Phase 1 manifest referenced by Phase 2',
    observed_phase1_hash == m2['phase1_manifest_sha256'],
    f"expected={m2['phase1_manifest_sha256']}; observed={observed_phase1_hash}"
)
check('Phase 1 source record identity', m1.get('source_record') == EXPECTED_SOURCE_RECORD, m1.get('source_record'))
check('Phase 1 raw-files-preserved status', bool(m1.get('raw_files_preserved')), m1.get('raw_files_preserved'))

# Rebuild the search roots around both manifests. This covers a canonical repository,
# a flat Downloads folder, or a folder tree containing browser-suffixed copies.
DEPENDENCY_ROOTS = unique_paths([
    *dependency_search_roots(manifest2_path, SEARCH_ROOTS),
    *dependency_search_roots(manifest1_path, SEARCH_ROOTS)
])

all_dependency_names = set(m1['export_sha256']) | set(m2['export_sha256']) | set(m1['input_sha256'])
candidate_index = build_candidate_index(all_dependency_names, DEPENDENCY_ROOTS, max_depth=6)

phase1_paths = {}
phase2_paths = {}
raw_paths = {}
resolution_notes = {}

for name, expected in m1['export_sha256'].items():
    path, actual, note = choose_by_expected_hash(name, expected, candidate_index.get(name, []))
    phase1_paths[name] = path
    resolution_notes[canonical_logical_path('Phase 1', name)] = note

for name, expected in m2['export_sha256'].items():
    path, actual, note = choose_by_expected_hash(name, expected, candidate_index.get(name, []))
    phase2_paths[name] = path
    resolution_notes[canonical_logical_path('Phase 2', name)] = note

for name, expected in m1['input_sha256'].items():
    path, actual, note = choose_by_expected_hash(name, expected, candidate_index.get(name, []))
    raw_paths[name] = path
    resolution_notes[canonical_logical_path('Raw', name)] = note

verified_path_hashes = {}
verified_path_labels = {}
verification_rows = []

def register_verified(stage, canonical_name, path, expected_hash, actual_hash, status, note):
    logical = canonical_logical_path(stage, canonical_name)
    verified_path_hashes[path] = actual_hash
    verified_path_labels[path] = logical
    verification_rows.append({
        'stage': stage,
        'logical_path': logical,
        'resolved_filename': path.name,
        'expected_sha256': expected_hash,
        'actual_sha256': actual_hash,
        'status': status,
        'resolution_note': note,
    })

register_verified(
    'Phase 2 manifest', EXPECTED_PHASE2_NAME, manifest2_path, '', observed_manifest_hash,
    'RECORDED_NOT_PREPINNED',
    MANIFEST_DISCOVERY_METHOD + '; Phase 2 did not provide an independent upstream hash of its own manifest.'
)
register_verified(
    'Phase 1 manifest', EXPECTED_PHASE1_NAME, manifest1_path, m2['phase1_manifest_sha256'],
    observed_phase1_hash, 'PASS', phase1_manifest_resolution
)

for stage, mapping, hashes in [
    ('Phase 1', phase1_paths, m1['export_sha256']),
    ('Phase 2', phase2_paths, m2['export_sha256']),
    ('Raw', raw_paths, m1['input_sha256'])
]:
    for name, expected in hashes.items():
        path = mapping[name]
        actual = sha(path)
        check(
            'SHA-256: ' + stage + '/' + name,
            actual == expected,
            f'file={path.name}; expected={expected}; observed={actual}'
        )
        register_verified(
            stage, name, path, expected, actual, 'PASS',
            resolution_notes[canonical_logical_path(stage, name)]
        )

logical_hashes = {
    verified_path_labels[path]: digest
    for path, digest in verified_path_hashes.items()
}
for stored_path, expected in m2['verified_input_sha256'].items():
    relative = stored_path.replace('\\', '/')
    check(
        'Phase 2 input reference: ' + relative,
        logical_hashes.get(relative) == expected,
        f'expected={expected}; observed={logical_hashes.get(relative, "not verified")}'
    )

t1 = {name: read_csv(phase1_paths[name]) for name in m1['export_sha256']}
t2 = {name: read_csv(phase2_paths[name]) for name in m2['export_sha256']}

check('Phase 1 export coverage', len(t1) == 11, len(t1))
check('Phase 2 export coverage', len(t2) == 26 and set(t2) == set(m2['rows']), len(t2))

for name, expected in m2['rows'].items():
    check(
        'Phase 2 row count: ' + name,
        len(t2[name]) == int(expected),
        f'expected={expected}; observed={len(t2[name])}'
    )

EXPECTED_PHASE2_CORE_COUNTS = {
    'worksheet_register.csv': 92,
    'measurement_points.csv': 4600,
    'measurement_values_source.csv': 39100,
    'image_records_source.csv': 1031,
    'validation_summary.csv': 43,
    'value_qc_findings.csv': 24,
}
for name, expected in EXPECTED_PHASE2_CORE_COUNTS.items():
    check(
        'Phase 2 checkpoint invariant: ' + name,
        len(t2[name]) == expected,
        f'expected={expected}; observed={len(t2[name])}'
    )

check(
    'Recorded Phase 2 checks passed',
    t2['validation_summary.csv']['status'].eq('PASS').all(),
    t2['validation_summary.csv']['status'].value_counts().to_dict()
)

# Choose the output folder only after the controlled inputs have been resolved.
output_env = os.environ.get('HYDROGEL_BIOINK_OUTPUT_DIR', '').strip()
if OUTPUT_ROOT_OVERRIDE is not None:
    OUT = Path(OUTPUT_ROOT_OVERRIDE).expanduser().resolve()
    OUTPUT_LOCATION_MODE = 'OUTPUT_ROOT_OVERRIDE'
elif output_env:
    OUT = Path(output_env).expanduser().resolve()
    OUTPUT_LOCATION_MODE = 'HYDROGEL_BIOINK_OUTPUT_DIR'
elif REPOSITORY_ROOT is not None:
    OUT = REPOSITORY_ROOT / 'data/processed/standardisation_and_quality_checks'
    OUTPUT_LOCATION_MODE = 'canonical repository data/processed folder'
else:
    OUT = manifest2_path.parent / 'phase3_outputs/standardisation_and_quality_checks'
    OUTPUT_LOCATION_MODE = 'portable fallback beside the resolved Phase 2 checkpoint'

input_verification = pd.DataFrame(verification_rows)
display(input_verification[['stage', 'logical_path', 'resolved_filename', 'status', 'resolution_note']])
display(
    input_verification.groupby(['stage', 'status'], dropna=False)
    .size().rename('verified_files').reset_index()
)
print('All controlled dependencies were resolved by manifest identity and/or SHA-256.')
print('Phase 2 manifest SHA-256 recorded for this run:', observed_manifest_hash)
print('Phase 3 output mode:', OUTPUT_LOCATION_MODE)
print('Phase 3 output folder:', OUT)

# Exact schemas inspected from the supplied, verified checkpoint.
EXPECTED_SCHEMAS = {'entity_schema.csv': ['table', 'row_granularity', 'primary_key', 'parent_scope', 'scientific_boundary'], 'source_header_inventory.csv': ['source_header', 'occurrences', 'blank_cells'], 'image_index_column_profile.csv': ['column', 'rows', 'blank_count', 'blank_percent', 'distinct_nonblank'], 'value_qc_findings.csv': ['rule', 'severity', 'source', 'location', 'evidence', 'action'], 'phase2_issue_decisions.csv': ['phase1_issue_id', 'issue_code', 'phase2_status', 'decision'], 'audit_issue_log.csv': ['issue_id', 'issue_code', 'severity', 'source', 'sheet', 'column', 'evidence', 'next_action'], 'validation_summary.csv': ['check', 'status', 'evidence'], 'missing_value_rules.csv': ['source_condition', 'interpretation', 'action'], 'image_missingness.csv': ['source_column', 'rows', 'unknown_blank'], 'phase1_issue_carry_forward.csv': ['issue_id', 'issue_code', 'severity', 'source', 'sheet', 'column', 'evidence', 'next_action'], 'strain_conversion_preview.csv': ['member_path', 'sheet', 'source_excel_row', 'source_column_position', 'source_header', 'source_value_json', 'source_unit', 'numeric_value', 'parse_status', 'conversion_factor', 'standardised_value', 'standardised_unit', 'conversion_rule', 'transformation_status'], 'measurement_points.csv': ['member_path', 'sheet', 'source_excel_row'], 'worksheet_register.csv': ['member_path', 'sheet', 'source_stem', 'member_sha256', 'measurement_family', 'header_row', 'point_count', 'physical_sample_link_status'], 'context_review.csv': ['question', 'observed', 'interpretation'], 'measurement_values_source.csv': ['member_path', 'sheet', 'source_excel_row', 'source_column_position', 'source_header', 'source_value_json', 'source_unit', 'numeric_value', 'parse_status'], 'image_field_dictionary.csv': ['source_column', 'logical_type', 'definition', 'semantic_limit'], 'data_dictionary.csv': ['table', 'field', 'logical_type', 'definition', 'provenance', 'unit_policy', 'missingness_policy'], 'image_filename_expectations.csv': ['source_csv_record', 'image_id', 'expected_filename', 'existence_status'], 'source_manifest.csv': ['source_record', 'source_version', 'metadata_review_date', 'canonical_filename', 'selected_filename', 'relative_path', 'bytes', 'md5', 'reference_md5', 'checksum_match', 'sha256', 'dataset_licence_status'], 'measurement_context_summary.csv': ['measurement_family', 'source_header', 'worksheets', 'minimum', 'maximum', 'fewest_distinct_values', 'most_distinct_values'], 'quality_control_rules.csv': ['rule', 'severity', 'criterion', 'action'], 'deferred_entities.csv': ['entity', 'reason_not_instantiated'], 'workbook_structure_inventory.csv': ['member_path', 'sheet', 'reported_max_row', 'reported_max_column', 'candidate_header_row', 'data_rows', 'blank_data_rows', 'duplicate_rows', 'formula_cells', 'excel_error_cells', 'headers_json'], 'image_index_value_counts.csv': ['column', 'source_value', 'is_blank', 'count'], 'input_verification.csv': ['relative_path', 'sha256', 'status'], 'measurement_column_profile.csv': ['member_path', 'sheet', 'column_position', 'excel_column', 'source_header', 'observed_rows', 'blank_count', 'blank_percent', 'numeric_count', 'nonnumeric_nonblank_count', 'nonfinite_numeric_count', 'distinct_nonblank', 'numeric_min', 'numeric_max', 'numeric_median'], 'archive_member_inventory.csv': ['member_path', 'is_directory', 'folder', 'suffix', 'bytes', 'compressed_bytes', 'crc32', 'sha256'], 'additional_review_items.csv': ['review_topic', 'status', 'next_action'], 'image_observed_categories.csv': ['column', 'source_value', 'is_blank', 'count'], 'literature_evidence_register.csv': ['document', 'source_location', 'source_statement', 'scope', 'schema_decision'], 'measurement_axis_profile.csv': ['member_path', 'sheet', 'source_header', 'numeric_min', 'numeric_max', 'distinct_numeric_values'], 'linkage_summary.csv': ['quantity', 'count'], 'filename_candidates_unconfirmed.csv': ['source_stem', 'GG_count', 'viscosity_count', 'GG_members', 'viscosity_members', 'exact_one_each', 'physical_sample_link'], 'candidate_filename_correspondences.csv': ['source_stem', 'GG_count', 'viscosity_count', 'GG_members', 'viscosity_members', 'exact_one_each', 'physical_sample_link'], 'source_to_schema_mapping.csv': ['source_header', 'canonical_field', 'logical_type', 'source_unit', 'conversion_policy', 'definition'], 'reviewed_inputs.csv': ['reviewed_filename', 'sha256', 'role'], 'image_records_source.csv': ['source_csv_record', 'image_id', 'date', 'ink_type_ph', 'shape', 'intensity', 'completeness_sam', 'contain_sps', 'image_id_status', 'date_status', 'ink_type_ph_status', 'shape_status', 'intensity_status', 'completeness_sam_status', 'contain_sps_status', 'expected_filename', 'existence_status']}

for tables in [t1, t2]:
    for filename, table in tables.items():
        expected = EXPECTED_SCHEMAS[filename]
        check(
            'Schema: ' + filename,
            list(table.columns) == expected,
            f'expected={expected}; observed={list(table.columns)}'
        )

## 3. Preserve the evidence boundary and transformation decisions

**Data-science concept.** An evidence register separates source statements from interpretations and pending decisions.

**Materials and rheology concept.** Apparent steady-shear viscosity and oscillatory complex viscosity are different observables. The paper reports amplitude sweeps at 1 Hz; this does not assign an LVR to each exported curve.

**Why this matters.** A familiar-looking unit token is insufficient to approve its conversion.

**What the code does.** Record the reviewed document hashes, the method location and decisions for strain, viscosity, image annotations and source context. Supporting documents are evidence used during notebook construction; they are not runtime dependencies for Phase 3 execution.

**How to interpret the output.** Supported mappings can proceed; unresolved meanings stay explicit.

**What this can and cannot establish.** The paper and supplementary captions do not provide an explicit export-token definition in the reviewed text. No claim is made that all possible instrument documentation has been exhausted.


In [ ]:
evidence_register = pd.DataFrame([
 {'evidence_id':'DOC_METHOD','source':'Main publication, DOI 10.1080/17452759.2026.2671497',
  'locator':'Section 5.1; PDF page 15 / printed page 14',
  'sha256':'c3868eb63b4f4ce96be0b39ea4b058cc91ccf86d98697ff4ca673bfb560b6536',
  'observation':'HAAKE MARS III; nominal shear-rate range 0.01–1000 s^-1; amplitude sweeps at 1 Hz and 22.5 °C.',
  'boundary':'Method context does not replace recorded conditions or define the export token mPas.'},
 {'evidence_id':'DOC_SUPPLEMENT','source':'Supplied supplementary document (reviewed copy)',
  'locator':'Supplementary text and Figure S2 caption',
  'sha256':'16bfe75b0ddf0be5029d384fed2f888ffdad9c0fe80e2803850d0b1cd2f98e81',
  'observation':'Supplementary captions describe fitted parameters and model evaluation; no export-token definition was identified in the reviewed text.',
  'boundary':'A viscosity label in a fitted-parameter figure would not alone establish raw export notation.'},
 {'evidence_id':'SOURCE_HEADERS','source':'Pinned raw rheology ZIP and Phase 2 source mapping',
  'locator':'Original worksheet header cells; checked in Section 4', 'sha256':m1['input_sha256']['ALG-Ph_HA-Ph_rheology_data.zip'],
  'observation':'Exact strain headers distinguish percent from source-labelled fraction.',
  'boundary':'The conversion respects labels; it cannot establish that the instrument labels were scientifically correct.'}
])
decisions = pd.DataFrame([
 ['STRAIN','supported_by_header','Divide explicitly labelled percent by 100; retain labelled fractions.','SOURCE_HEADERS'],
 ['VISCOSITY','unresolved','Retain mPas and source numeric values; standardised viscosity value/unit remain blank.','DOC_METHOD;DOC_SUPPLEMENT'],
 ['IMAGE_ANNOTATIONS','unresolved','Retain literal annotations and unknown blanks; do not infer outcomes or reagent presence.','Phase 2 image dictionary'],
 ['INTENSITY','unresolved','Retain the source token; do not convert to physical irradiance.','Phase 2 image dictionary'],
 ['ACQUISITION_CONTEXT','source_preserved','Keep recorded conditions attached through exact point coordinates.','SOURCE_HEADERS;DOC_METHOD'],
 ['PHYSICAL_LINKAGE','deferred_phase4','No physical samples, formulations or replicate IDs are assigned.','Phase 2 linkage summary'],
 ['LVR_AND_FITTING','out_of_scope','Do not choose an LVR, fit curves or derive representative material properties.','DOC_METHOD']
],columns=['topic','decision_status','decision','evidence_reference'])
display(decisions)


## 4. Check source coordinates against the original workbooks

**Data-science concept.** Natural keys preserve location: archive member, worksheet, Excel row and column. Foreign-key checks establish source nesting.

**Materials and rheology concept.** A curve contains measurements at different acquisition conditions; the worksheet is a source container, not a verified sample.

**Why this matters.** This independently confirms that the exported scalar and header belong to the original cell before conversion.

**What the code does.** Check primary and foreign keys, member hashes, headers, row coverage and every source scalar against the raw ZIP. Verify the image-index fields against the original CSV.

**How to interpret the output.** All 39,100 source cells must reconcile exactly with the Phase 2 serialisation.

**What this can and cannot establish.** This validates extraction fidelity, including original header tokens; it does not resolve formulation or replicate identity.

### Measurement hierarchy and statistical independence

| Level | Existing natural key | Meaning and boundary |
| --- | --- | --- |
| Worksheet | `member_path`, `sheet` | One source worksheet; no confirmed physical sample |
| Measurement point | Above plus `source_excel_row` | One recorded row under particular acquisition conditions |
| Source cell | Above plus `source_column_position` | One response, condition or acquisition label |
| Image-index record | `source_csv_record` | One CSV record; image label is retained separately |

A **natural key** uses existing source coordinates to distinguish records. A **foreign key** points from a child record to its parent. **Referential integrity** means every child has the expected parent. **Join cardinality** describes how many matches a join permits: joining a cell to its point should be many-to-one, never an uncontrolled many-to-many expansion.

A **nested observation** belongs to a larger group. Points along a curve share source context and may be strongly related. **Pseudoreplication** occurs when such related observations are treated as independent experimental replicates. The 39,100 cells include acquisition labels and conditions, not 39,100 independent materials. Even 92 worksheets cannot be called 92 physical samples without evidence.

In later modelling, **data leakage** could occur if related points from the same material or run appear in both training and testing data: the model may effectively see part of the test material during training. Phase 3 preserves the hierarchy needed to investigate appropriate grouping later. It does not create train/test splits or solve physical identity.

**Permitted assumption:** source-coordinate matches establish record provenance. **Not permitted:** similar filenames establish shared samples. These checks prepare Phase 4 without performing its scientific linkage.


In [ ]:
values = t2['measurement_values_source.csv'].copy()
points = t2['measurement_points.csv'].copy()
worksheets = t2['worksheet_register.csv'].copy()
images = t2['image_records_source.csv'].copy()
mapping = t2['source_to_schema_mapping.csv'].copy()
WK = ['member_path','sheet']
PK = WK + ['source_excel_row']
CK = PK + ['source_column_position']
for label,table,key in [('worksheet',worksheets,WK),('point',points,PK),('cell',values,CK),
                        ('image record',images,['source_csv_record'])]:
    check(label+' keys nonblank',table[key].ne('').all().all())
    check(label+' keys unique',not table.duplicated(key).any())
check('Points have worksheet parents',len(points.merge(worksheets[WK],on=WK,how='inner',validate='many_to_one'))==len(points))
check('Cells have point parents',len(values.merge(points[PK],on=PK,how='inner',validate='many_to_one'))==len(values))
check('Header mapping unique',not mapping.source_header.duplicated().any())
check('Header mapping coverage',set(values.source_header)==set(mapping.source_header))
lookup = values.set_index(CK)
source_headers = mapping.set_index('source_header').source_unit.to_dict()
verified_cells = 0
with zipfile.ZipFile(raw_paths['ALG-Ph_HA-Ph_rheology_data.zip']) as archive:
    check('Raw workbook coverage',set(n for n in archive.namelist() if n.lower().endswith('.xlsx'))==set(worksheets.member_path))
    for member,group in worksheets.groupby('member_path',sort=False):
        payload = archive.read(member)
        check('Member hash: '+member,group.member_sha256.eq(hashlib.sha256(payload).hexdigest()).all())
        wb = openpyxl.load_workbook(io.BytesIO(payload),read_only=True,data_only=False)
        try:
            check('Sheet coverage: '+member,set(wb.sheetnames)==set(group.sheet))
            for w in group.itertuples(index=False):
                rows = list(wb[w.sheet].iter_rows())
                check('Header row: '+member,int(w.header_row)==1)
                headers = ['' if c.value is None else str(c.value) for c in rows[0]]
                active = [(i,row) for i,row in enumerate(rows[1:],2)
                          if not all(c.value is None or (isinstance(c.value,str) and not c.value.strip()) for c in row)]
                check('Raw point coverage: '+member,len(active)==int(w.point_count))
                for row_number,row in active:
                    for col,(header,cell) in enumerate(zip(headers,row),1):
                        key = (member,w.sheet,str(row_number),str(col))
                        if key not in lookup.index:
                            raise ValueError('Unrepresented source cell: '+str(key))
                        r = lookup.loc[key]
                        if cell.data_type in {'f','e'} or r.source_header != header or r.source_unit != source_headers[header]:
                            raise ValueError('Source header/type mismatch: '+str(key))
                        if r.source_value_json != json.dumps(cell.value,ensure_ascii=False,allow_nan=False):
                            raise ValueError('Source scalar mismatch: '+str(key))
                        verified_cells += 1
        finally:
            wb.close()
check('All raw cells reconciled',verified_cells==len(values),verified_cells)
with raw_paths['image_index.csv'].open(encoding='utf-8-sig',newline='') as f:
    reader = csv.DictReader(f); raw_image_fields = reader.fieldnames; raw_images = list(reader)
check('Raw image count',len(raw_images)==len(images))
check('Image dictionary coverage',set(raw_image_fields)==set(t2['image_field_dictionary.csv'].source_column))
for i,raw in enumerate(raw_images,2):
    record = images.loc[images.source_csv_record.eq(str(i))]
    if len(record)!=1 or any(record.iloc[0][field]!=raw[field] for field in raw_image_fields):
        raise ValueError('Image source mismatch at CSV record '+str(i))
print(f'Reconciled {verified_cells:,} source cells and {len(images):,} image records.')


## 5. Define and apply traceable value transformations

**Data-science concept.** An explicit rule table makes a transformation inspectable and reproducible. Original columns are retained; derived columns are added.

**Materials and rheology concept.** Percent strain and fractional strain differ by a factor of 100. Pa, Pa·s and Hz represent different physical quantities.

**Why this matters.** Canonical field names alone must not conceal an unresolved unit or an altered scalar.

**What the code does.** Build one rule per exact observed header; check numeric parsing against the preserved JSON scalar; apply identity and percent-to-fraction rules only.

**How to interpret the output.** A deferred viscosity row retains its source number but has no approved standardised viscosity number. SegIndex remains a text acquisition label.

**What this can and cannot establish.** Converting a representation does not validate the instrument calibration or repair unusual values.

### Standardisation is not statistical scaling

Here **standardisation** means representing scientifically equivalent information consistently: canonical fields, supported units and explicit statuses. It is not z-score normalisation or feature scaling. **Unit harmonisation** is valid only after both the quantity and unit meaning are known. A shared dimensional unit cannot make different observables interchangeable.

**Viscoelasticity** means that a material exhibits both elastic energy storage and viscous dissipation. In oscillatory rheology, G′ (storage modulus) describes the elastic contribution and G″ (loss modulus) describes the dissipative contribution under the applied conditions; both are in Pa. Comparing an isolated pair does not by itself prove gelation, printability or an appropriate measurement regime.

**Apparent viscosity** describes the flow response under steady-shear conditions and may depend on shear rate. **Complex viscosity magnitude**, |η\*|, belongs to an oscillatory response. They must remain distinct even if their units are eventually harmonised. Their equivalence would require additional justification; Phase 3 imposes no such relationship.

The header-supported viscosity mappings retain `mPas`. The publication presents viscosity quantities in Pa·s, but that is insufficient to define a raw export token. The current decision is to preserve the source number and token, leave unsupported converted values blank and explain the reason in the transformation status. No conversion factor is approved for these rows.

**Permitted assumption:** a rule can use the verified header mapping. **Not permitted:** an unfamiliar or surprising unit can be guessed. A deferred transformation is a valid outcome and remains visible to future users.


In [ ]:
rule_rows = []
for i,r in enumerate(mapping.itertuples(index=False),1):
    policy = r.conversion_policy
    check('Known conversion policy: '+r.source_header,policy in {'retain','identity','divide by 100','deferred'})
    target = ('1 (strain fraction)' if r.canonical_field=='strain_fraction' else r.source_unit) if policy in {'identity','divide by 100'} else ''
    rule_rows.append({'rule_id':f'TR{i:02d}','source_header':r.source_header,'canonical_field':r.canonical_field,
        'source_unit':r.source_unit,'operation':policy,'factor':0.01 if policy=='divide by 100' else 1.0 if policy=='identity' else np.nan,
        'target_unit':target,'evidence':'Exact source header; Phase 2 mapping',
        'scientific_caution':('Retained as labelled fraction; unusual values are not reinterpreted as percent.' if r.source_header=='ɣ in -' else 'Unit token unresolved; no Pa·s value approved.' if policy=='deferred' else 'Representation only; no physical-sample inference.'),
        'approval_status':'deferred' if policy=='deferred' else 'text_retention' if policy=='retain' else 'supported_by_header'})
rules = pd.DataFrame(rule_rows)
standardised = values.merge(rules.drop(columns='source_unit'),on='source_header',how='left',validate='many_to_one',sort=False)
parsed = []
for r in standardised.itertuples(index=False):
    scalar = json.loads(r.source_value_json)
    if r.operation=='retain':
        parsed.append(np.nan); continue
    if scalar is None or (isinstance(scalar,str) and not scalar.strip()):
        status,num='blank_unknown',np.nan
    elif isinstance(scalar,bool):
        status,num='invalid_boolean',np.nan
    else:
        try:
            num=float(scalar); status='numeric' if np.isfinite(num) else 'nonfinite'
        except (TypeError,ValueError):
            status,num='nonnumeric',np.nan
    if status!=r.parse_status:
        raise ValueError('Parse status differs from checkpoint: '+str((r.member_path,r.sheet,r.source_excel_row,r.source_column_position)))
    if status=='numeric' and not np.isclose(num,float(r.numeric_value),rtol=1e-14,atol=0):
        raise ValueError('Parsed scalar differs from checkpoint numeric value.')
    parsed.append(num)
standardised['source_numeric_checked']=parsed
eligible=standardised.operation.isin(['identity','divide by 100']) & standardised.parse_status.eq('numeric')
standardised['standardised_value']=np.nan
standardised.loc[eligible,'standardised_value']=standardised.loc[eligible,'source_numeric_checked']*standardised.loc[eligible,'factor']
standardised['standardised_unit']=standardised.target_unit
standardised['transformation_status']=np.select(
    [standardised.operation.eq('deferred'),standardised.operation.eq('retain'),eligible],
    ['deferred_unit_semantics','text_preserved','applied'],default='not_applied_parse_review')
standardised=standardised.merge(worksheets[WK+['measurement_family']],on=WK,validate='many_to_one',sort=False)
check('Cell count preserved after transformation',len(standardised)==len(values))
check('Original columns preserved',standardised[values.columns].equals(values))
check('No viscosity conversion',standardised.loc[standardised.operation.eq('deferred'),'standardised_value'].isna().all())
check('Finite standardised values',np.isfinite(standardised.loc[eligible,'standardised_value']).all())
display(standardised.groupby(['canonical_field','transformation_status'],sort=False).size().rename('source_cells').reset_index())


## 6. Validate the strain conversion against the Phase 2 preview

**Data-science concept.** Independent arithmetic checks and inverse transformations catch scale errors; preview reconciliation checks continuity.

**Materials and rheology concept.** A strain fraction of 0.01 equals 1%. Large source-labelled fractions are preserved, not capped at one.

**Why this matters.** A factor-of-100 error would change the apparent deformation regime.

**What the code does.** Compare every strain cell by source key with the preview, check the explicit percent and fraction counts, and reverse the conversion within a stated floating-point tolerance.

**How to interpret the output.** Expect 100 percent-labelled and 2,200 fraction-labelled strain cells.

**What this can and cannot establish.** These tests establish faithful labelled conversion, not an LVR or correct instrument labelling.

### What strain represents and why a factor of 100 matters

**Strain** measures relative deformation. For simple shear, it relates lateral displacement to the separation over which that displacement occurs; it is dimensionless. In oscillatory testing, strain amplitude describes the magnitude of that periodic deformation. A percentage and a fraction encode that magnitude on different numerical scales: an illustrative 1% becomes 0.01 as a fraction, and 0.01 multiplied by 100 returns 1%.

The actual checkpoint contains exact headers `ɣ in %` and `ɣ in -`. We apply percentage conversion only to the first. The second is retained numerically under the Phase 2 source-labelled fraction interpretation. A dash can denote a dimensionless quantity, but the semantic interpretation of unusual large values remains uncertain; the notebook does not secretly divide them by 100 to make them look plausible.

A **round-trip check** applies the inverse transformation and compares with the original numerical value. Computers store most decimal fractions approximately, so comparison uses declared relative and absolute tolerances rather than demanding exact binary equality. This test detects implementation and scaling errors; it cannot prove that the original unit label is correct.

An **amplitude sweep** changes strain amplitude, whereas a **frequency sweep** changes oscillation frequency. The **linear viscoelastic region (LVR)** is a regime in which response is approximately proportional to deformation and the moduli are approximately amplitude-independent under the specified test conditions. A source header, fixed frequency or arithmetic conversion does not determine the LVR. Selecting it requires later scientific analysis.

**Permitted assumption:** explicitly labelled percent uses a factor of 0.01. **Not permitted:** large dimensionless values imply percent or an invalid measurement. This protects later deformation-dependent interpretation.


In [ ]:
strain = standardised.loc[standardised.canonical_field.eq('strain_fraction')].copy()
preview = t2['strain_conversion_preview.csv']
check('Strain preview key uniqueness',not preview.duplicated(CK).any())
joined = strain.merge(preview[CK+['standardised_value','conversion_factor']],on=CK,how='outer',
                      suffixes=('','_preview'),indicator=True,validate='one_to_one')
check('Strain preview coverage',joined['_merge'].eq('both').all())
check('Strain arithmetic agrees with preview',np.allclose(joined.standardised_value.astype(float),joined.standardised_value_preview.astype(float),rtol=1e-12,atol=1e-15))
check('Strain factors agree with preview',np.allclose(joined.factor.astype(float),joined.conversion_factor.astype(float),rtol=0,atol=0))
check('Percent strain count',strain.source_header.eq('ɣ in %').sum()==100)
check('Fraction strain count',strain.source_header.eq('ɣ in -').sum()==2200)
check('Strain round trip',np.allclose(strain.standardised_value/strain.factor,strain.source_numeric_checked,rtol=1e-12,atol=1e-15))
display(strain.groupby(['source_header','source_unit','standardised_unit']).size().rename('cells').reset_index())


## 7. Preserve image semantics and validate dates

**Data-science concept.** Missingness is a data state rather than a negative label. A date can be standardised syntactically without changing its source meaning.

**Materials and rheology concept.** An unrecorded reagent annotation does not establish reagent absence. A shape label is not a print-quality measurement.

**Why this matters.** Automatic Boolean or numeric conversion could manufacture training labels.

**What the code does.** Retain all Phase 2 image columns, verify blank statuses, and add an ISO date only for strictly valid YYYY/MM/DD dates.

**How to interpret the output.** Unknown blanks remain blank with unknown\_blank status. Date output describes format only.

**What this can and cannot establish.** No physical image inspection, intensity conversion, formulation parsing or annotation outcome inference occurs.

### Missingness is scientific information

| Representation | What can be established |
| --- | --- |
| Numeric zero | A recorded numeric value, not a missing observation |
| Blank source token | An observed blank; this checkpoint calls its meaning unknown |
| Literal `no` | A reported annotation token; its full scientific scope still needs evidence |
| Quantity absent from a worksheet | Structural absence; no source cell is invented |
| Not measured / not reported / not applicable | Distinct explanations, used only when the source supports them |
| Unavailable image file | Its existence/content has not been checked in this workflow |
| Blank standardised viscosity value | Conversion deferred; consult the retained source value and status |

Missing-data handling can otherwise create false labels. A blank `contain_sps` does not establish absence of the reagent, and a blank `completeness_sam` does not prove failure or success. An intensity token also cannot become physical irradiance without a validated field definition and calibration.

The ISO date is only a consistent representation of a valid source date. No timezone or event label is invented. The reverse format check verifies information preservation, not whether the date refers to printing, imaging or another event.

**Permitted assumption:** strict calendar parsing can validate representation. **Not permitted:** date syntax establishes event meaning. Preserving these distinctions prevents misleading future classification targets.


In [ ]:
image_standardised=images.copy()
for field in raw_image_fields:
    blank=images[field].str.strip().eq('')
    check('Image blank status: '+field,images[field+'_status'].eq('unknown_blank').equals(blank))
shape_ok=images.date.str.fullmatch(r'\d{4}/\d{2}/\d{2}')
parsed_dates=pd.to_datetime(images.date.where(shape_ok),format='%Y/%m/%d',errors='coerce')
image_standardised['date_iso']=parsed_dates.dt.strftime('%Y-%m-%d').fillna('')
image_standardised['date_transformation_status']=np.where(shape_ok & parsed_dates.notna(),'format_standardised','unresolved_date')
check('Image original columns preserved',image_standardised[images.columns].equals(images))
print('Valid date formats:',int(image_standardised.date_transformation_status.eq('format_standardised').sum()))
display(pd.DataFrame([{'field':f,'unknown_blanks':int(images[f+'_status'].eq('unknown_blank').sum())} for f in raw_image_fields]))

valid_dates = shape_ok & parsed_dates.notna()
check('Date round trip', parsed_dates.loc[valid_dates].dt.strftime('%Y/%m/%d').equals(images.loc[valid_dates,'date']))
image_standardised['date_conversion_rule']='DATE01: strict YYYY/MM/DD to YYYY-MM-DD; no event semantics inferred'


## 8. Link existing QC findings and rerun value checks

**Data-science concept.** A finding table and a separate association table allow one finding to affect multiple observations without pretending they are independent findings.

**Materials and rheology concept.** Nonpositive rheology values need measurement-context review. They are retained even when unsuitable for logarithms or later fitting.

**Why this matters.** The 24 Phase 2 findings include two column-level image missingness findings; they are not 24 samples.

**What the code does.** Assign documented provenance IDs based on finding row order, resolve exact cell locations, associate image blank findings with blank records, and rerun numeric sign checks. Add review links for shear rates below the nominal method range.

**How to interpret the output.** Finding counts and association counts differ. A source cell can have several review reasons.

**What this can and cannot establish.** Location parsing establishes provenance linkage only. It does not identify physical samples or diagnose the cause of negative values.

### Validation, quality control and uncertainty

**Validation** asks whether data satisfy a defined rule. **Cleaning** changes data. A **QC finding** records an observation or group that meets a review criterion. A flag does not, by itself, mean a bad row, failed experiment or permission to repair it.

For example, a negative viscosity value requires review before logarithms or fitting. Its sign alone does not identify an instrument limitation, export issue or other cause. We retain both the measurement and its conditions, so the evidence remains available for investigation. No arbitrary upper limit is imposed to remove unusual strain values.

The 24 inherited findings include 20 nonpositive-response findings, two acquisition-axis findings and two image-column missingness findings. A **finding-to-observation association** records which source cells or image fields are affected. Expanding a column-level finding to its affected records increases association count, not independent sample count. Generated `P2-QC-...` IDs are technical finding references based on the pinned table’s row order, not invented sample IDs.

The publication’s nominal shear-rate lower limit is compared with source values as a review criterion. It is not a filter that forces the raw data to agree with prose. Negative-rate records can correctly carry both an acquisition-axis finding and a document-range finding.

**Permitted assumption:** exact recorded coordinates can associate a finding with its observation. **Not permitted:** a flag reveals the physical cause or resolves a sample relationship. Structured findings support later human review and transparent decisions.


In [ ]:
prior=t2['value_qc_findings.csv'].copy()
prior.insert(0,'finding_id',[f'P2-QC-{i:03d}' for i in range(1,len(prior)+1)])
prior['finding_origin']='Phase 2 value_qc_findings.csv; one-based data-record order'
links=[]
for r in prior.itertuples(index=False):
    if r.rule=='IMAGE_BLANK':
        check('Image finding field',r.location in raw_image_fields,r.location)
        matches=images.loc[images[r.location].str.strip().eq('')]
        for im in matches.itertuples(index=False):
            links.append({'finding_id':r.finding_id,'target_type':'image_field','member_path':'','sheet':'',
              'source_excel_row':'','source_column_position':'','source_csv_record':im.source_csv_record,'image_field':r.location})
    else:
        match=re.fullmatch(r'(.*); row (\d+); column (\d+); (.*)',r.location)
        check('Parse exact QC source location: '+r.finding_id,match is not None)
        sheet,row,col,header=match.groups()
        hit=values.loc[values.member_path.eq(r.source)&values.sheet.eq(sheet)&values.source_excel_row.eq(row)&values.source_column_position.eq(col)]
        check('QC location resolves: '+r.finding_id,len(hit)==1 and hit.iloc[0].source_header==header)
        links.append({'finding_id':r.finding_id,'target_type':'measurement_cell','member_path':r.source,'sheet':sheet,
                      'source_excel_row':row,'source_column_position':col,'source_csv_record':'','image_field':''})
qc_links=pd.DataFrame(links)
check('All Phase 2 findings linked',set(qc_links.finding_id)==set(prior.finding_id))
num=standardised.source_numeric_checked
response=standardised.source_header.isin(["G' in Pa",'G" in Pa','η in mPas','|η*| in mPas'])
axis_bad=(standardised.source_header.isin(['f in Hz','ɣ̇ in 1/s']) & num.le(0)) | (standardised.canonical_field.eq('strain_fraction') & num.lt(0))
for rule,mask in [('NONPOSITIVE_RESPONSE',response & num.le(0)),('ACQUISITION_AXIS',axis_bad)]:
    expected_ids=prior.loc[prior.rule.eq(rule),'finding_id']
    expected=qc_links.loc[qc_links.finding_id.isin(expected_ids),CK]
    actual=standardised.loc[mask,CK]
    check('Recomputed locations: '+rule,set(map(tuple,expected.to_numpy()))==set(map(tuple,actual.to_numpy())),len(actual))
# A document-range difference is a review flag, not an exclusion rule.
below=standardised.loc[standardised.source_header.eq('ɣ̇ in 1/s') & num.lt(0.01)]
new_findings=pd.DataFrame([{'finding_id':'P3-QC-001','rule':'BELOW_NOMINAL_METHOD_RANGE','severity':'REVIEW',
    'source':'Main publication Section 5.1 versus observed source cells','location':'shear-rate cells < 0.01 1/s',
    'evidence':f'{len(below)} cells; nominal lower bound is 0.01 1/s',
    'action':'Retain all observations; do not impose the publication range as a filter.',
    'finding_origin':'Phase 3 document-context review'}])
new_links=[{'finding_id':'P3-QC-001','target_type':'measurement_cell',**dict(zip(CK,k)),
            'source_csv_record':'','image_field':''} for k in below[CK].itertuples(index=False,name=None)]
qc_links=pd.concat([qc_links,pd.DataFrame(new_links)],ignore_index=True)
qc_findings=pd.concat([prior,new_findings],ignore_index=True)
check('QC association uniqueness',not qc_links.duplicated().any())
cell_ids=qc_links.loc[qc_links.target_type.eq('measurement_cell')].groupby(CK,sort=False).finding_id.agg(lambda x:';'.join(sorted(set(x)))).rename('qc_finding_ids').reset_index()
standardised=standardised.merge(cell_ids,on=CK,how='left',validate='one_to_one',sort=False)
standardised['qc_finding_ids']=standardised.qc_finding_ids.fillna('')
standardised['qc_status']=np.where(standardised.qc_finding_ids.ne(''),'flagged_for_review','no_flag_from_applied_value_checks')
image_ids=qc_links.loc[qc_links.target_type.eq('image_field')].groupby('source_csv_record').finding_id.agg(lambda x:';'.join(sorted(set(x)))).rename('qc_finding_ids').reset_index()
image_standardised=image_standardised.merge(image_ids,on='source_csv_record',how='left',validate='one_to_one',sort=False)
image_standardised['qc_finding_ids']=image_standardised.qc_finding_ids.fillna('')
check('QC additions preserve source cells',standardised[values.columns].equals(values))
check('QC additions preserve image records',image_standardised[images.columns].equals(images))
display(qc_findings.groupby(['finding_origin','rule']).size().rename('finding_records').reset_index())
print('QC associations:',len(qc_links),'— these are provenance links, not samples.')


## 9. Keep acquisition conditions attached and check the final contract

**Data-science concept.** Foreign keys and an observation-preserving view keep conditions alongside responses without many-to-many joins.

**Materials and rheology concept.** Frequency, strain, temperature, stress and time change the interpretation of rheological responses. Nominal method settings must not replace observed values.

**Why this matters.** A clean-looking table can become scientifically misleading if it loses acquisition axes.

**What the code does.** Create a point view from the long table using exact source keys, retaining source-valued viscosity and acquisition labels. Recheck counts, nesting, duplicates and context coverage.

**How to interpret the output.** The point view has one row per original measurement point. Missing columns across test families mean structural absence in this view, not zero.

**What this can and cannot establish.** This is a convenience view; the long table remains authoritative for original values, headers, units, QC and transformation lineage. No physical linkage is implied.

### Conditions are part of a rheological measurement

**Shear rate** expresses the rate of deformation in steady flow, in inverse seconds. **Frequency** counts oscillation cycles per second, in Hz. These are not interchangeable despite both involving time. Angular frequency is a related oscillatory quantity, but no angular-frequency field is manufactured here. Phase angle is not an observed field in this checkpoint and is not inferred.

The verified source cells include recorded temperature, frequency, strain, shear rate, time, segment time and stress where present. Temperature and deformation can change polymer-chain motion and interactions; shear and processing history can influence observed flow response. Geometry and nominal protocol statements from the publication remain study-level context and are not inserted as if each source row measured them.

The point view is a **pivot**: it changes the table layout while retaining one row per verified source point. A uniqueness check precedes pivoting so aggregation cannot silently hide repeated fields. The source-cell table remains authoritative because it distinguishes an actual blank from a quantity absent from a test layout and records each transformation.

**Permitted assumption:** exact point keys preserve conditions beside responses. **Not permitted:** a common temperature or frequency proves material comparability. Future structure–property or formulation–process–performance work will still need supported material identity and comparable protocols.


In [ ]:
check('One canonical field per source point',not standardised.duplicated(PK+['canonical_field']).any())
view=standardised[PK+['canonical_field']].copy()
view['view_value']=[r.standardised_value if r.transformation_status=='applied' else
                    json.loads(r.source_value_json) for r in standardised.itertuples(index=False)]
point_view=view.pivot(index=PK,columns='canonical_field',values='view_value').reset_index()
point_view.columns.name=None
point_view=points.merge(point_view,on=PK,how='left',validate='one_to_one',sort=False)
point_view=point_view.merge(worksheets[WK+['measurement_family','physical_sample_link_status']],on=WK,validate='many_to_one',sort=False)
point_view['viscosity_unit_status']='mPas retained; conversion unresolved'
point_view['observation_unit']='source measurement point; physical sample unconfirmed'
check('Point-view coverage',len(point_view)==len(points)==EXPECTED_PHASE2_CORE_COUNTS['measurement_points.csv'],len(point_view))
check('Source-cell coverage',len(standardised)==EXPECTED_PHASE2_CORE_COUNTS['measurement_values_source.csv'],len(standardised))
check('Image coverage',len(image_standardised)==EXPECTED_PHASE2_CORE_COUNTS['image_records_source.csv'],len(image_standardised))
check('Physical links remain unresolved',point_view.physical_sample_link_status.eq('unresolved').all())
for field in ['frequency_hz','strain_fraction','temperature_c','shear_rate_per_s','time_s','segment_time_s','shear_stress_pa']:
    expected=int(standardised.canonical_field.eq(field).sum())
    actual=int(point_view[field].notna().sum())
    check('Point-view context coverage: '+field,actual==expected,actual)
# Duplicate source rows are reviewed within a worksheet; no rows are dropped.
row_signatures=values.groupby(PK,sort=False).source_value_json.agg(tuple).reset_index(name='signature')
duplicate_points=row_signatures.loc[row_signatures.duplicated(WK+['signature'],keep=False)].drop(columns='signature')
check('No unexplained duplicate full rows in pinned snapshot',len(duplicate_points)==0,len(duplicate_points))
check('Dates valid in pinned snapshot',image_standardised.date_transformation_status.eq('format_standardised').all())
check('Retained negative response cells',int((response & num.le(0)).sum())==20)
check('Retained nonpositive acquisition-axis cells',int(axis_bad.sum())==2)
check('Image blank counts retained',images.completeness_sam.eq('').sum()==943 and images.contain_sps.eq('').sum()==922)
context_coverage=standardised.groupby(['measurement_family','canonical_field','source_unit','standardised_unit'],dropna=False).size().rename('source_cells').reset_index()
display(context_coverage)
print('All observations retained. No physical-sample count has been established.')


## 10. Export the Phase 3 handoff and verify preservation

**Data-science concept.** A manifest records input hashes, rule versions, output hashes, row counts and runtime. Export read-back checks protect the handoff.

**Materials and rheology concept.** Standardisation preserves measurement meaning only when its limitations travel with the values.

**Why this matters.** Future linkage work needs both reusable tables and visible unresolved decisions.

**What the code does.** Write declared Phase 3 outputs, verify their row counts and hashes, and confirm every input remains byte-identical.

**How to interpret the output.** A successful run ends with a passed validation ledger and an output manifest. Timestamps can differ across runs; deterministic data outputs should not.

**What this can and cannot establish.** The handoff supports Phase 4 source-led linkage. It does not establish model readiness or permission to redistribute third-party data.

### Reproducibility across sessions

**Idempotence** means that repeating the processing does not keep adding or changing records. This notebook replaces only named Phase 3 outputs. It never appends another copy of a dataset and never reads a previous Phase 3 output to recreate its scientific state.

**Determinism** means that the same controlled inputs and documented environment produce the same logical results. Source order is preserved, joins declare cardinality, and finding references have a defined order. The UTC run time is deliberately variable metadata; it is excluded when comparing scientific content across runs.

All state is reconstructed by the notebook’s earlier cells. The required sequence is **save, close, reopen, choose the documented kernel, restart, run all**. Discovery no longer depends on the exact repository folder name. A candidate file may have a browser suffix such as `(1)` or `(3)`, but it is accepted only when the upstream manifest hash confirms byte identity. Conflicting checkpoints stop the run. Provenance exported by Phase 3 uses canonical logical paths while `input_verification.csv` records the locally resolved filename, so machine-specific absolute paths are not required for scientific traceability.

A successful run proves only the implemented contracts. Unresolved viscosity notation, strain semantics, physical identity, image meanings, the 45 fitted-data versus 46 source-file population difference and redistribution questions remain documented. Good future feature engineering needs that uncertainty, rather than a falsely complete dataset.

**Permitted assumption:** verified inputs are stable for the duration of the run. **Not permitted:** a successful software run certifies experimental validity, independent observations or public redistribution rights.


In [ ]:
# Original QC/history tables remain at their verified Phase 2 paths; no need to duplicate them.
unresolved = t2['additional_review_items.csv'].copy()
unresolved['phase3_disposition'] = [
    'Conversion remains deferred; see evidence and decision registers.' if topic == 'VISCOSITY_UNIT_TOKEN' else
    'Retained as an open question; no LVR selection, population inference or forced exclusion.'
    for topic in unresolved.review_topic
]

# Re-hash every protected input immediately before export. This works whether the local
# filename is canonical or browser-suffixed because provenance is keyed by the canonical
# logical path and the accepted bytes were hash-verified earlier.
for path, original in verified_path_hashes.items():
    logical = verified_path_labels[path]
    check('Input preserved: ' + logical, sha(path) == original, path.name)

issue_decisions = t2['phase2_issue_decisions.csv'].copy()
issue_decisions['phase3_decision'] = [
    'Source-label strain standardisation implemented and validated; unusual fraction semantics remain open.'
    if code == 'MIXED_STRAIN_UNITS' else
    'Prior decision retained; no new scientific interpretation or forced closure.'
    for code in issue_decisions.issue_code
]
check(
    'Phase 1 issue history preserved',
    len(issue_decisions) == 15 and
    set(issue_decisions.phase1_issue_id) == set(t2['phase1_issue_carry_forward.csv'].issue_id)
)

exports = {
    'input_verification.csv': input_verification,
    'document_evidence_register.csv': evidence_register,
    'standardisation_decisions.csv': decisions,
    'phase3_issue_decisions.csv': issue_decisions,
    'transformation_rules.csv': rules,
    'measurement_values_standardised.csv': standardised,
    'measurement_points_view.csv': point_view,
    'image_records_standardised.csv': image_standardised,
    'qc_findings.csv': qc_findings,
    'qc_observation_links.csv': qc_links,
    'acquisition_context_coverage.csv': context_coverage,
    'unresolved_review_items.csv': unresolved,
}

OUT.mkdir(parents=True, exist_ok=True)
for name, table in exports.items():
    target = OUT / name
    temporary = target.with_suffix('.csv.tmp')
    table.to_csv(temporary, index=False, encoding='utf-8-sig', lineterminator='\n')
    check('Export row count: ' + name, len(read_csv(temporary)) == len(table), len(table))
    temporary.replace(target)

check(
    'Protected inputs unchanged after export',
    all(sha(path) == digest for path, digest in verified_path_hashes.items())
)

validation_summary = pd.DataFrame(checks)
exports['validation_summary.csv'] = validation_summary
validation_summary.to_csv(
    OUT / 'validation_summary.csv',
    index=False,
    encoding='utf-8-sig',
    lineterminator='\n'
)

input_sha256_logical = {
    verified_path_labels[path]: digest
    for path, digest in verified_path_hashes.items()
}
input_resolution = {
    verified_path_labels[path]: {
        'resolved_filename': path.name,
        'resolution_note': next(
            (row['resolution_note'] for row in verification_rows
             if row['logical_path'] == verified_path_labels[path]),
            ''
        )
    }
    for path in verified_path_hashes
}

manifest = {
    'phase': '3: standardisation and quality checks',
    'notebook': '03_standardisation_and_quality_checks.ipynb',
    'run_utc': datetime.now(timezone.utc).isoformat(),
    'source_record': m2['source_record'],
    'source_version': m2['source_version'],
    'phase2_manifest_sha256': sha(manifest2_path),
    'phase1_manifest_sha256': m2['phase1_manifest_sha256'],
    'input_sha256': input_sha256_logical,
    'input_resolution': input_resolution,
    'environment': runtime.to_dict('records'),
    'input_discovery_mode': SEARCH_MODE,
    'phase2_manifest_discovery': MANIFEST_DISCOVERY_METHOD,
    'output_location_mode': OUTPUT_LOCATION_MODE,
    'phase2_manifest_hash_status': 'recorded at Phase 3 runtime; no independent upstream self-hash was available',
    'rows': {name: len(table) for name, table in exports.items()},
    'export_sha256': {name: sha(OUT / name) for name in exports},
    'input_rows': {name: len(table) for name, table in t2.items()},
    'transformation_status_counts': {
        str(k): int(v) for k, v in standardised.transformation_status.value_counts().items()
    },
    'qc_rule_counts': {str(k): int(v) for k, v in qc_findings.rule.value_counts().items()},
    'qc_associations': len(qc_links),
    'validation_counts': {
        str(k): int(v) for k, v in validation_summary.status.value_counts().items()
    },
    'unresolved_decisions': decisions.loc[
        decisions.decision_status.isin(['unresolved', 'deferred_phase4'])
    ].to_dict('records'),
    'rules_version': 'phase3-v2-content-addressed-input-resolution',
    'physical_sample_linkage': 'unresolved; Phase 4 deferred',
    'viscosity_conversion': 'deferred; original mPas retained',
    'strain_conversion': 'explicit source percent divided by 100; source fraction retained',
    'image_blanks': 'unknown; preserved',
    'raw_and_checkpoints_preserved': True,
    'dataset_redistribution': 'unresolved; source-derived exports for local review',
    'model_readiness': 'not established'
}

target = OUT / 'standardisation_run_manifest.json'
temporary = target.with_suffix('.json.tmp')
temporary.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2, allow_nan=False),
    encoding='utf-8'
)
temporary.replace(target)

display(pd.DataFrame([{'file': name, 'rows': len(table)} for name, table in exports.items()]))
print(f'{len(validation_summary)} checks passed; {len(exports)} CSVs and one JSON manifest exported.')
print('Output folder:', OUT)

## Handoff interpretation and next step

The outputs retain one long-table row per source cell, one convenience-view row per source point and one image row per original CSV record. `qc_observation_links.csv` links findings to exact source coordinates. Its derived finding IDs identify QC records, never physical samples. The original 24 Phase 2 findings remain distinguishable from the new document-range review finding; expanded image blank associations do not increase the number of independent findings or materials.

`measurement_values_standardised.csv` retains **every original Phase 2 column**. Additional columns record the canonical field, rule ID, operation, factor, target unit, evidence, approval status, independently parsed source number, standardised value/unit, transformation status, measurement family and QC finding IDs/status. An empty standardised viscosity value means **conversion deferred**, not missing source measurement. Use `numeric_value`, `source_value_json` and `source_unit` for the retained source representation.

The point view contains fraction-valued strain where conversion was supported; its viscosity fields end in `_source` and retain `mPas`. Conditions remain at their original source points. Fields absent from a test family are blank in that convenience view; inspect the long table to distinguish structural absence from an observed blank. The image table preserves all original tokens/status fields and adds only date-format and QC provenance fields. No inferred concentrations or outcome labels are added.

The verified Phase 2 files remain the authoritative history for all 15 Phase 1 findings, their Phase 2 decisions, dictionaries and filename candidates. Their hashes are recorded in the Phase 3 manifest. The four additional review topics remain open: viscosity notation, LVR selection, publication-range differences and model-population reconciliation.

**Next manageable step:** review this run's validation summary and unresolved decisions. Full record linkage belongs to conceptual Phase 4 (`04_record_linkage.ipynb` in the revised phase numbering). Do not proceed to fitting or machine learning merely because structural checks pass.

### References and evidence limits

* Source publication (2026). *Generative AI-guided in silico closed-loop optimisation of deposition morphology for 3D bioprinting applications*. [DOI: 10.1080/17452759.2026.2671497](https://doi.org/10.1080/17452759.2026.2671497), especially Section 5.1. The supplied PDF hash is recorded above.
* Supplied supplementary document: reviewed textual captions, including Figure S2. The document hash is recorded above. No raw-export unit definition is claimed from those captions.
* [Zenodo source record 19602891](https://zenodo.org/records/19602891), pinned to the Phase 1/2 snapshot, version `v0.0.2`.
* Completed Phase 2 schema, dictionaries, source mappings, QC findings and manifests supplied with this project.

Instrument-specific export documentation or a source-author clarification could justify a later viscosity conversion. Until such evidence is recorded, `mPas` remains unchanged. No internet access or external documentation is required to execute this notebook.
